In [1]:
import requests
import datetime
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import deque
import threading
from tqdm import tqdm
import queue

In [2]:
# API Key
API_KEY = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'

In [3]:
# API Rate Limit Constants
API_LIMIT = 3000  # Max API calls per minute
WINDOW_SIZE = 60  # Seconds (1 minute)
MAX_WORKERS = 250  # Dynamically adjusted
MAX_RETRIES = 3  # Maximum retry attempts
BACKOFF_FACTOR = 1.5  # Exponential backoff factor

# Track API request timestamps
request_times = deque()
request_lock = threading.Lock()
request_queue = queue.Queue()


def rate_limit():
    """Strictly enforces the 3,000 API calls per minute limit."""
    global request_times

    with request_lock:
        now = time.time()

        # Remove timestamps older than 60 seconds
        while request_times and now - request_times[0] > WINDOW_SIZE:
            request_times.popleft()

        # If at the limit, wait until a slot opens
        while len(request_times) >= API_LIMIT:
            wait_time = WINDOW_SIZE - (now - request_times[0])
            print(f"⚠️ Rate limit reached! Sleeping for {wait_time:.2f} seconds...")
            time.sleep(wait_time)
            now = time.time()
            while request_times and now - request_times[0] > WINDOW_SIZE:
                request_times.popleft()

        # Register the new API request timestamp
        request_times.append(time.time())


def fetch_data_with_retries(url):
    """Fetches data from API with automatic retries and strict rate limiting."""
    for attempt in range(1, MAX_RETRIES + 1):
        rate_limit()

        try:
            response = requests.get(url)
            if response.status_code == 200:
                return response.json()
            else:
                print(f"🚨 Attempt {attempt}: API returned {response.status_code}. Retrying...")
        except requests.exceptions.RequestException as e:
            print(f"🔴 Attempt {attempt}: Network error: {e}. Retrying...")

        # Exponential backoff before retrying
        time.sleep(BACKOFF_FACTOR ** attempt)

    print(f"❌ Failed after {MAX_RETRIES} attempts: {url}")
    return None


def get_all_tickers():
    """Fetches a list of all investable stocks and indexes from FMP API."""
    stock_url = f"https://financialmodelingprep.com/api/v3/stock/list?apikey={API_KEY}"
    index_url = f"https://financialmodelingprep.com/api/v3/symbol/available-indexes?apikey={API_KEY}"

    stock_response = fetch_data_with_retries(stock_url)
    index_response = fetch_data_with_retries(index_url)

    if not stock_response or not index_response:
        print("⚠️ Failed to retrieve stock/index lists.")
        return []

    stock_tickers = [item['symbol'] for item in stock_response]
    index_tickers = [item['symbol'] for item in index_response]

    return stock_tickers + index_tickers


def get_sd(ticker):
    """Fetches standard deviation for a stock over 1Y, 5Y, and 10Y."""
    base_url = 'https://financialmodelingprep.com/api/v3/technical_indicator/1day/'
    periods = {"1Y": 252, "5Y": 1260, "10Y": 2520}
    sd_values = {}

    for key, period in periods.items():
        url = f"{base_url}{ticker}?type=standardDeviation&period={period}&apikey={API_KEY}"
        request_queue.put(url)  # Add to queue
        data = fetch_data_with_retries(url)

        if isinstance(data, list) and len(data) > 0:
            sd_values[key] = round(data[0].get('standardDeviation', None), 4)
        else:
            sd_values[key] = None

    return sd_values


def get_cagr(ticker):
    """Fetches CAGR for a stock over 1Y, 5Y, and 10Y."""
    base_url = 'https://financialmodelingprep.com/api/v3/historical-price-full/'
    years_list = [1, 5, 10]
    cagr_values = {}

    for years in years_list:
        end_date = datetime.date.today()
        start_date = end_date - datetime.timedelta(days=years * 365)

        url = f"{base_url}{ticker}?from={start_date}&to={end_date}&apikey={API_KEY}"
        request_queue.put(url)  # Add to queue
        data = fetch_data_with_retries(url)

        if data and "historical" in data and len(data["historical"]) > 0:
            historical_data = sorted(data["historical"], key=lambda x: x["date"])
            P_start = historical_data[0]["close"]
            P_end = historical_data[-1]["close"]

            cagr = ((P_end / P_start) ** (1 / years)) - 1
            cagr_values[f"{years}Y"] = round(cagr * 100, 2)
        else:
            cagr_values[f"{years}Y"] = None

    return cagr_values


def get_stock_data(ticker):
    """Combines Standard Deviation and CAGR data into a single dictionary per stock."""
    try:
        sd_data = get_sd(ticker)
        cagr_data = get_cagr(ticker)

        return {
            "Ticker": ticker,
            "1Y SD": sd_data["1Y"],
            "5Y SD": sd_data["5Y"],
            "10Y SD": sd_data["10Y"],
            "1Y CAGR": cagr_data["1Y"],
            "5Y CAGR": cagr_data["5Y"],
            "10Y CAGR": cagr_data["10Y"]
        }
    except Exception as e:
        print(f"❌ Error processing {ticker}: {e}")
        return None


def get_multiple_stocks_data(tickers):
    """Fetches SD and CAGR for multiple stocks using threading while respecting rate limits."""
    total_tickers = len(tickers)
    all_data = []

    print(f"🚀 Processing {total_tickers} tickers using {MAX_WORKERS} threads...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(get_stock_data, ticker): ticker for ticker in tickers}
        
        for future in tqdm(as_completed(futures), total=total_tickers, desc="📈 Processing Stocks"):
            result = future.result()
            if result:
                all_data.append(result)

    df = pd.DataFrame(all_data)
    df.to_csv("all_stocks_sd_cagr.csv", index=False)

    return df

In [4]:
all_tickers = get_all_tickers()
df = get_multiple_stocks_data(all_tickers)
print(df.head())

🚀 Processing 85102 tickers using 250 threads...


📈 Processing Stocks:  72%|███████▏  | 61473/85102 [8:10:10<5:47:02,  1.13it/s] 

❌ Error processing ARSC: float division by zero


📈 Processing Stocks: 100%|██████████| 85102/85102 [11:17:31<00:00,  2.09it/s]  


   Ticker   1Y SD   5Y SD  10Y SD  1Y CAGR  5Y CAGR  10Y CAGR
0  VCT.NZ  0.1197  0.2650  0.3883     9.81     6.35      3.40
1   KFIIR     NaN     NaN     NaN      NaN      NaN      0.00
2  MEL.NZ  0.2916  0.6710  1.3610    -3.83     5.91      7.72
3  NZG.NZ  0.0816     NaN     NaN     1.67    -0.48     -0.24
4  SKC.NZ  0.1916  0.6439  0.9520   -30.32   -11.18    -10.70
